# Building an MCP Server with GitHub Integration

This notebook demonstrates how to build, run, and test a Model Context Protocol (MCP) server that integrates with the GitHub API.

## What You'll Learn

- What MCP is and how the protocol works (JSON-RPC over stdio/SSE)
- How to create an MCP server with tools and resources using `FastMCP`
- How to test the server with an MCP client
- How to configure the server for different deployment modes (stdio, SSE, Docker)
- How an LLM would interact with the server via tool calling

In [ ]:
from __future__ import annotations

import os
import sys
import json
import subprocess
import time
import threading
from pathlib import Path

# Check if MCP package is installed
try:
    import mcp
    from mcp.server.fastmcp import FastMCP
    print(f"✅ MCP package installed (version: {mcp.__version__ if hasattr(mcp, '__version__') else 'unknown'})")
except ImportError:
    print("❌ MCP package not installed. Run: pip install mcp httpx")
    sys.exit(1)

import httpx
print("✅ httpx installed")
print(f"Python: {sys.version}")

## 1. Understanding MCP Architecture

MCP is built on JSON-RPC 2.0. The three core concepts are:

| Concept | Purpose | Lifecycle |
|---------|---------|-----------|
| **Tools** | Functions the LLM can call | Called on demand during a conversation |
| **Resources** | Data exposed with URI scheme | Accessed when the LLM needs data |
| **Prompts** | Reusable prompt templates | Loaded by the host |

**Communication flow:**
```
Host (Cline, Claude)                                  Server
      │                                                  │
      │────── initialize (capabilities negotiation) ────→│
      │←───── initialized ───────────────────────────────│
      │                                                  │
      │────── tools/list ───────────────────────────────→│
      │←───── [tool definitions] ────────────────────────│
      │                                                  │
      │────── tools/call (name, arguments) ─────────────→│
      │←───── tool result ───────────────────────────────│
```

Each message is a JSON-RPC request/response:
```json
// Request
{"jsonrpc":"2.0","id":1,"method":"tools/call","params":{"name":"search_repositories","arguments":{"query":"machine learning"}}}

// Response
{"jsonrpc":"2.0","id":1,"result":{"content":[{"type":"text","text":"...results..."}]}}
```

## 2. Building the GitHub MCP Server

We'll use `FastMCP` — the quickest way to build an MCP server. Each `@mcp.tool()` decorator registers a tool that the LLM can discover and call.

The server below provides five tools:
1. `search_repositories` — Search GitHub repos by query
2. `get_repository` — Get detailed info about a specific repo
3. `list_issues` — List issues for a repo
4. `get_readme` — Fetch the README content
5. `create_issue` — Create a new issue (needs GITHUB_TOKEN)

And one resource:
- `github://{owner}/{repo}/issues` — Expose open issues as a readable resource

In [ ]:
# Define the MCP server IN THIS NOTEBOOK

GITHUB_API_BASE = "https://api.github.com"
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN", "")

HEADERS = {"Accept": "application/vnd.github.v3+json"}
if GITHUB_TOKEN:
    HEADERS["Authorization"] = f"Bearer {GITHUB_TOKEN}"

# Create the server instance
server = FastMCP("GitHub Assistant")

# ──────────────────────────────────────────────
# Tool 1: Search repositories
# ──────────────────────────────────────────────
@server.tool()
async def search_repositories(query: str, per_page: int = 5) -> str:
    """Search for GitHub repositories matching a query.
    
    Args:
        query: The search query (e.g. "quantum computing python")
        per_page: Number of results to return (max 30)
    """
    async with httpx.AsyncClient() as client:
        response = await client.get(
            f"{GITHUB_API_BASE}/search/repositories",
            params={"q": query, "per_page": per_page, "sort": "stars"},
            headers=HEADERS,
        )
        response.raise_for_status()
        data = response.json()
    
    if not data["items"]:
        return "No repositories found."
    
    results = []
    for repo in data["items"]:
        results.append(
            f"• {repo['full_name']} — ⭐ {repo['stargazers_count']} — "
            f"{repo['description'] or 'No description'}"
        )
    return "\n".join(results)


# ──────────────────────────────────────────────
# Tool 2: Get repository details
# ──────────────────────────────────────────────
@server.tool()
async def get_repository(owner: str, repo: str) -> str:
    """Get detailed information about a specific repository.
    
    Args:
        owner: Repository owner (user or organization)
        repo: Repository name
    """
    async with httpx.AsyncClient() as client:
        response = await client.get(
            f"{GITHUB_API_BASE}/repos/{owner}/{repo}",
            headers=HEADERS,
        )
        response.raise_for_status()
        data = response.json()
    
    return (
        f"📦 {data['full_name']}\n"
        f"📝 {data['description'] or 'No description'}\n"
        f"⭐ Stars: {data['stargazers_count']}  🍴 Forks: {data['forks_count']}\n"
        f"🐛 Open Issues: {data['open_issues_count']}\n"
        f"📅 Created: {data['created_at'][:10]}  "
        f"🔄 Updated: {data['updated_at'][:10]}\n"
        f"🌐 {data['html_url']}"
    )


# ──────────────────────────────────────────────
# Tool 3: List issues
# ──────────────────────────────────────────────
@server.tool()
async def list_issues(owner: str, repo: str, state: str = "open", per_page: int = 5) -> str:
    """List issues for a GitHub repository.
    
    Args:
        owner: Repository owner
        repo: Repository name
        state: Issue state (open, closed, all)
        per_page: Number of issues to return
    """
    async with httpx.AsyncClient() as client:
        response = await client.get(
            f"{GITHUB_API_BASE}/repos/{owner}/{repo}/issues",
            params={"state": state, "per_page": per_page, "sort": "updated"},
            headers=HEADERS,
        )
        response.raise_for_status()
        issues = response.json()
    
    if not issues:
        return f"No {state} issues found."
    
    results = []
    for issue in issues:
        if "pull_request" not in issue:  # Filter out PRs
            results.append(
                f"• #{issue['number']} {issue['title']} — "
                f"by {issue['user']['login']} ({issue['state']})"
            )
    return "\n".join(results) if results else f"No {state} issues found (excluding PRs)."


# ──────────────────────────────────────────────
# Tool 4: Get README
# ──────────────────────────────────────────────
@server.tool()
async def get_readme(owner: str, repo: str) -> str:
    """Get the README content of a repository (truncated to 2000 chars).
    
    Args:
        owner: Repository owner
        repo: Repository name
    """
    async with httpx.AsyncClient() as client:
        response = await client.get(
            f"{GITHUB_API_BASE}/repos/{owner}/{repo}/readme",
            headers={**HEADERS, "Accept": "application/vnd.github.raw+json"},
        )
        response.raise_for_status()
        return response.text[:2000]


# ──────────────────────────────────────────────
# Tool 5: Create an issue (requires token)
# ──────────────────────────────────────────────
@server.tool()
async def create_issue(owner: str, repo: str, title: str, body: str = "") -> str:
    """Create a new issue on a repository.
    
    Requires GITHUB_TOKEN to be set.
    
    Args:
        owner: Repository owner
        repo: Repository name
        title: Issue title
        body: Issue description (optional)
    """
    if not GITHUB_TOKEN:
        return "❌ GITHUB_TOKEN not set. Cannot create issues."
    
    async with httpx.AsyncClient() as client:
        response = await client.post(
            f"{GITHUB_API_BASE}/repos/{owner}/{repo}/issues",
            headers=HEADERS,
            json={"title": title, "body": body},
        )
        response.raise_for_status()
        data = response.json()
    
    return f"✅ Issue created: {data['html_url']}"


# ──────────────────────────────────────────────
# Resource: Expose issues as a URI-addressable resource
# ──────────────────────────────────────────────
@server.resource("github://{owner}/{repo}/issues")
async def get_issues_resource(owner: str, repo: str) -> str:
    """Open issues as a readable resource."""
    return await list_issues(owner, repo, state="open", per_page=10)


print("✅ Server defined with 5 tools and 1 resource")
print(f"   GITHUB_TOKEN set: {bool(GITHUB_TOKEN)}")

## 3. Testing the Server with a Low-Level MCP Client

To test the server without starting a full subprocess, we'll use the MCP client library to connect to it programmatically. This simulates what an MCP host (like Cline) does internally.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Save the server code to a temporary file
SERVER_FILE = Path("./github_mcp_server_demo.py")

server_code = '''
import os
import sys
import httpx
from mcp.server.fastmcp import FastMCP

GITHUB_API_BASE = "https://api.github.com"
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN", "")
HEADERS = {"Accept": "application/vnd.github.v3+json"}
if GITHUB_TOKEN:
    HEADERS["Authorization"] = f"Bearer {GITHUB_TOKEN}"

server = FastMCP("GitHub Assistant")

@server.tool()
async def search_repositories(query: str, per_page: int = 3) -> str:
    """Search GitHub repos."""
    async with httpx.AsyncClient() as client:
        resp = await client.get(
            f"{GITHUB_API_BASE}/search/repositories",
            params={"q": query, "per_page": per_page, "sort": "stars"},
            headers=HEADERS,
        )
        resp.raise_for_status()
        data = resp.json()
    if not data["items"]:
        return "No results."
    return "\n".join(f"\u2022 {r['full_name']} \u2b50 {r['stargazers_count']}" for r in data["items"])

@server.tool()
async def get_repository(owner: str, repo: str) -> str:
    """Get repo details."""
    async with httpx.AsyncClient() as client:
        resp = await client.get(f"{GITHUB_API_BASE}/repos/{owner}/{repo}", headers=HEADERS)
        resp.raise_for_status()
        d = resp.json()
    return f"{d['full_name']} \u2b50 {d['stargazers_count']} \ud83c\udf4d {d['forks_count']} \ud83d\udc1b {d['open_issues_count']}"

if __name__ == "__main__":
    server.run()
'''

SERVER_FILE.write_text(server_code)
print(f"✅ Server script saved to {SERVER_FILE}")

In [ ]:
async def test_mcp_server():
    """Connect to the MCP server via stdio and test the tools."""
    print("=" * 60)
    print("Starting MCP server and testing tools...")
    print("=" * 60)
    
    # Server parameters: run the Python script via stdio
    server_params = StdioServerParameters(
        command="python",
        args=[str(SERVER_FILE)],
        env={"GITHUB_TOKEN": GITHUB_TOKEN, "PYTHONUNBUFFERED": "1"},
    )
    
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # Step 1: Initialize the connection
            await session.initialize()
            print("\n✅ Connection initialized")
            
            # Step 2: List available tools
            tools_result = await session.list_tools()
            print(f"\n📋 Available tools ({len(tools_result.tools)}):")
            for tool in tools_result.tools:
                print(f"   • {tool.name}: {tool.description}")
                if tool.inputSchema:
                    props = tool.inputSchema.get("properties", {})
                    for p_name, p_info in props.items():
                        print(f"       - {p_name}: {p_info.get('description', '')}")
            
            # Step 3: Call search_repositories
            print("\n🔍 Testing: search_repositories(query='machine learning python')")
            result = await session.call_tool(
                "search_repositories",
                {"query": "machine learning python", "per_page": 3},
            )
            for content in result.content:
                if hasattr(content, "text"):
                    print(content.text)
            
            # Step 4: Call get_repository
            print("\n📦 Testing: get_repository(owner='tensorflow', repo='tensorflow')")
            result = await session.call_tool(
                "get_repository",
                {"owner": "tensorflow", "repo": "tensorflow"},
            )
            for content in result.content:
                if hasattr(content, "text"):
                    print(content.text)
            
            # Step 5: List resources
            print("\n📂 Available resources:")
            try:
                resources = await session.list_resources()
                for r in resources.resources:
                    print(f"   • {r.uri}: {r.description}")
            except Exception as e:
                print(f"   Resource listing: {e}")
    
    print("\n" + "=" * 60)
    print("✅ All tests completed")
    print("=" * 60)


# Run the test
import asyncio
await test_mcp_server()

## 4. Inspecting the JSON-RPC Messages

To understand what's happening under the hood, let's look at the raw JSON-RPC messages that flow between the host and server.

In [ ]:
# Simulate the JSON-RPC messages that flow over stdio

# A host sends this to list tools:
list_tools_request = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/list",
}
print("📤 Host → Server: tools/list")
print(json.dumps(list_tools_request, indent=2))

# The server responds with tool definitions:
list_tools_response = {
    "jsonrpc": "2.0",
    "id": 1,
    "result": {
        "tools": [
            {
                "name": "search_repositories",
                "description": "Search for GitHub repositories matching a query.",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string", "description": "Search query"},
                        "per_page": {"type": "number", "description": "Results to return"},
                    },
                    "required": ["query"],
                },
            },
        ]
    },
}
print("\n📥 Server → Host: tool definitions")
print(json.dumps(list_tools_response, indent=2))

# The host sends a tool call:
call_tool_request = {
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/call",
    "params": {
        "name": "search_repositories",
        "arguments": {"query": "quantum computing", "per_page": 2},
    },
}
print("\n📤 Host → Server: tools/call")
print(json.dumps(call_tool_request, indent=2))

## 5. Deployment Options

The server can be deployed in several modes. Each mode has different trade-offs.

### 5a. stdio Mode (Default)
The server runs as a subprocess. The host spawns it and communicates via stdin/stdout.

```bash
# Run directly
python github_mcp_server_demo.py

# Or via the mcp CLI
mcp run github_mcp_server_demo.py
```

### 5b. SSE Mode (HTTP Server)
The server runs as a web server on a port, accessible over HTTP.

```python
# Modify the last line to use SSE transport
# server.run(transport="sse", port=8000)
```

### 5c. Docker Deployment
Package the server in a Docker container for consistent deployment.

```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY github_mcp_server_demo.py .
EXPOSE 8000
CMD ["python", "github_mcp_server_demo.py"]
```

### 5d. Cloud Deployment (Railway, Fly.io, Render)
Run the SSE server in the cloud for persistent public access.

```bash
# railway.toml
[build]
  command = "pip install -r requirements.txt"
[deploy]
  startCommand = "python github_mcp_server_demo.py"
```

### 5e. GitHub Actions (CI/CD)
Run the MCP server as part of a GitHub Action workflow.

```yaml
name: MCP Test
on: [push]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
      - run: pip install mcp httpx
      - run: python github_mcp_server_demo.py &
      - run: sleep 2
      - run: python -c "from mcp.client.stdio import stdio_client; ..."
```

In [ ]:
# Demo: Show how to run in SSE mode (in a separate thread)
# This is for illustration - in practice you'd run this as a standalone process

print("=" * 60)
print("SSE (HTTP) Deployment Mode")
print("=" * 60)
print()
print("To run the server in SSE mode:")
print()
print("1. Create a deployment script:")
print()
deploy_code = '''
from mcp.server.fastmcp import FastMCP
import os, httpx

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN", "")
HEADERS = {"Accept": "application/vnd.github.v3+json"}
if GITHUB_TOKEN:
    HEADERS["Authorization"] = f"Bearer {GITHUB_TOKEN}"

server = FastMCP("GitHub Assistant")

@server.tool()
async def search_repositories(query: str, per_page: int = 5) -> str:
    """Search GitHub repos."""
    async with httpx.AsyncClient() as client:
        resp = await client.get(
            "https://api.github.com/search/repositories",
            params={"q": query, "per_page": per_page},
            headers=HEADERS,
        )
        return "\n".join(
            f"\u2022 {r['full_name']}" for r in resp.json().get("items", [])
        )

if __name__ == "__main__":
    # Run as SSE server on port 8000
    server.run(transport="sse", host="0.0.0.0", port=8000)
'''
print(deploy_code)
print()
print("2. Run the server:")
print("   export GITHUB_TOKEN=ghp_xxx")
print("   python deploy_sse.py")
print()
print("3. The server will be available at:")
print("   http://localhost:8000/sse")
print()
print("4. Configure your MCP host to connect:")
config = {
    "mcpServers": {
        "github-assistant": {
            "type": "url",
            "url": "http://localhost:8000/sse",
        }
    }
}
print(json.dumps(config, indent=2))

## 6. Simulating an LLM Using the MCP Server

This cell shows how an LLM would interact with the MCP server. The LLM receives tool descriptions, decides which tool to call, and the host executes the call and returns the result.

In [ ]:
# Simulate the LLM's decision process

# Step 1: The host presents tool descriptions to the LLM
tool_descriptions = """
Available tools:

1. search_repositories(query: str, per_page: int = 5)
   Search for GitHub repositories matching a query.

2. get_repository(owner: str, repo: str)
   Get detailed information about a specific repository.

3. list_issues(owner: str, repo: str, state: str = "open", per_page: int = 5)
   List issues for a GitHub repository.

4. get_readme(owner: str, repo: str)
   Get the README content of a repository.
"""

user_query = "Find the most popular machine learning library on GitHub and tell me about it."

print("🔍 User Query:")
print(f"   {user_query}")
print()
print("🧠 LLM Reasoning:")
print("   I need to:")
print("   1. Search for 'machine learning' repositories sorted by stars")
print("   2. Get details on the top result")
print()
print("📤 Tool Call 1: search_repositories(query='machine learning', per_page=1)")
print()
print("📥 Result: • tensorflow/tensorflow — ⭐ 188000")
print()
print("📤 Tool Call 2: get_repository(owner='tensorflow', repo='tensorflow')")
print()
print("📥 Result:")
print("   📦 tensorflow/tensorflow")
print("   📝 An Open Source Machine Learning Framework")
print("   ⭐ Stars: 188000  🍴 Forks: 72000")
print("   🐛 Open Issues: 2500")
print()
print("💬 Final Answer:")
print("   The most popular ML library is TensorFlow by Google, with 188K stars and 72K forks.")

## 7. Cleanup

Remove the temporary server file created during this notebook.

In [ ]:
# Cleanup temporary files
if SERVER_FILE.exists():
    SERVER_FILE.unlink()
    print(f"🗑️ Removed {SERVER_FILE}")

# Also clean up any deployment test files
for f in Path(".").glob("deploy_*.py"):
    f.unlink()
    print(f"🗑️ Removed {f}")

print("\n✅ Cleanup complete")

## Summary

In this notebook you:

1. **Understood MCP architecture** — hosts, servers, JSON-RPC protocol
2. **Built a GitHub MCP server** — 5 tools (search, get repo, list issues, readme, create issue) + 1 resource
3. **Tested the server** — connected via stdio client, listed tools, called tools
4. **Inspected JSON-RPC messages** — saw the raw request/response format
5. **Explored deployment options** — stdio, SSE, Docker, Cloud, GitHub Actions
6. **Simulated LLM interaction** — saw how an AI would use the tools to answer a question

### Next Steps

- Add more tools (e.g., `list_branches`, `create_pull_request`, `get_workflow_runs`)
- Expose more resources (e.g., `github://{owner}/{repo}/code/{path}` for file contents)
- Deploy the server to the cloud for persistent access
- Connect the server to your MCP host (Cline, Claude Desktop, etc.)